In [1]:
import pandas as pd 
import numpy as np 

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

## *Milestone 3 setup*

In [3]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 


print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 92.3 MB/s eta 0:00:00
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


## *Zero-shot classifier for Q1, Q2, Q6*

In [4]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

*Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)*

In [5]:
result = zs(prompt_150, candidate_labels=labels_150)

labels = result["labels"]
scores = result["scores"]

correct_index = labels.index(ans_150)
predicted_score = round(scores[correct_index], 3)

print(predicted_score)

0.384


*Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?*

In [6]:
prompt_embedding = model.encode([prompt_150])
distances, indices = index.search(prompt_embedding, k=10)

rank = list(indices[0]).index(150) + 1
print(rank)

10


## *Code to use a Cross-Encoder*

In [7]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in indices[0]]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

*Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?*

In [8]:
ranked_indices = [idx for score, idx in sorted(zip(ce_scores, indices[0]), reverse=True)]
rank = ranked_indices.index(150) + 1

print(rank)

1


*Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?*

In [9]:
from transformers import AutoTokenizer

row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

embedding = model.encode([prompt_42])
distances, indices = index.search(embedding, k=5)

retrieved_docs = [kb[idx] for idx in indices[0]]
concatenated_docs = " ".join(retrieved_docs)

input_text = f"Context: {concatenated_docs} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tokenizer.encode(input_text)

print(len(tokens))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


*Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).*

In [10]:
true_document = kb[150]
rag_prompt = f"Context: {true_document} Question: {prompt_150}"

result = zs(rag_prompt, candidate_labels=labels_150)

labels = result["labels"]
scores = result["scores"]

correct_index = labels.index(ans_150)
predicted_score = round(scores[correct_index], 3)

print(predicted_score)

0.989


*Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places)*

In [11]:
adversarial_document = kb[999]
adversarial_prompt = f"Context: {adversarial_document} Question: {prompt_150}"

result = zs(adversarial_prompt, candidate_labels=labels_150)

labels = result["labels"]
scores = result["scores"]

correct_index = labels.index(ans_150)
predicted_score = round(scores[correct_index], 3)

print(predicted_score)

0.529


*In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question*

*Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).*

In [12]:
hits = 0

for i in range(100):
    row = train.iloc[i]
    prompt = str(row["prompt"])
    ans = str(row[row["answer"]])

    embedding = model.encode([prompt])
    distances, indices = index.search(embedding, k=5)

    for idx in indices[0]:
        if ans in kb[idx]:
            hits += 1
            break

hit_rate = round((hits / 100) * 100, 1)
print(hit_rate)

73.0


*Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:*

*Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.*

*Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.*

*Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".*

*Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.*

*Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.*

*What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).*

In [13]:
aps = []

for i in range(20):
    row = train.iloc[i]
    prompt = str(row["prompt"])
    options = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"]),
    ]
    correct_text = str(row[row["answer"]])

    embedding = model.encode([prompt])
    distances, indices = index.search(embedding, k=5)
    retrieved_docs = [kb[idx] for idx in indices[0]]

    pairs = [[prompt, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = retrieved_docs[np.argmax(ce_scores)]

    rag_prompt = f"Context: {best_doc} Question: {prompt}"
    result = zs(rag_prompt, candidate_labels=options)
    pred_labels = result["labels"]

    rank = pred_labels.index(correct_text) + 1
    if rank == 1:
        ap = 1.0
    elif rank == 2:
        ap = 0.5
    elif rank == 3:
        ap = 1 / 3
    else:
        ap = 0.0

    aps.append(ap)

mean_map3 = round(sum(aps) / 20, 3)
print(mean_map3)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


0.975
